# PCA

```python
import pandas as pd 
import numpy as np
from PIL import Image #image 읽어오기 위한 라이브러리 
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submission.csv')

# PCA를 수행할 때 ID와 타겟의 정보는 불필요하므로 제거
train = train.drop(['ID', 'Outcome'], axis = 1)
test = test.drop(['ID'], axis = 1)
display(train.head())

# 예시 이미지 파일을 넘파이 배열로 변환
original_image = Image.open('car.jpg')
image_array = np.array(original_image)

# 원본 이미지 확인
plt.figure(figsize=(8, 4))
plt.imshow(image_array)
plt.axis('off')
plt.show()
```

## Image PCA 

```python
# RGB 채널 분리
r = image_array[:,:,0]
g = image_array[:,:,1]
b = image_array[:,:,2]

# 각 채널에 대해 PCA 수행
pca_R = PCA(n_components=0.8, svd_solver='full')
pca_G = PCA(n_components=0.8, svd_solver='full')
pca_B = PCA(n_components=0.8, svd_solver='full')

pca_R.fit(r)
pca_G.fit(g)
pca_B.fit(b)

R_transformed = pca_R.transform(r)
G_transformed = pca_G.transform(g)
B_transformed = pca_B.transform(b)

# 변환된 데이터로부터 각 채널 복원
R_restored = pca_R.inverse_transform(R_transformed)
G_restored = pca_G.inverse_transform(G_transformed)
B_restored = pca_B.inverse_transform(B_transformed)

# 복원된 채널을 결합하여 최종 이미지 생성
image_restored = np.stack((R_restored, G_restored, B_restored), axis=-1)

# 원본 이미지와 복원된 이미지 시각화 및 저장
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(image_array)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(image_restored.astype('uint8'))
ax[1].set_title('Restored Image')
ax[1].axis('off')

plt.show()
```

In [1]:
# PCA Function Example 
from sklearn.decomposition import PCA
import numpy as np

# 가상의 대용량 데이터 생성 (10,000행 x 500열)
X_dummy = np.random.rand(10000, 500)

# 1. 자동 선택 (가장 무난함)
pca_auto = PCA(n_components=10, svd_solver='auto')
X_auto = pca_auto.fit_transform(X_dummy)

# 2. 아주 정확한 계산이 필요할 때 (소형 데이터 권장)
pca_full = PCA(n_components=10, svd_solver='full')
X_full = pca_full.fit_transform(X_dummy)

# 3. 데이터가 매우 커서 강제적으로 빠르게 근사 계산하고 싶을 때
# (현재 질문하신 코드와 가장 관련 깊음)
pca_rand = PCA(n_components=10, svd_solver='randomized', random_state=42)
X_rand = pca_rand.fit_transform(X_dummy)

# PCA

- PCA(주성분 분석) 는 고차원의 데이터를 저차원으로 축소하는 통계적 기법입니다.

# 장점
- 데이터의 중요한 특성을 유지하며 차원을 줄이고, 데이터셋의 복잡성을 감소시킵니다.  
- 불필요한 변동성을 제거함으로써 데이터의 핵심 특성을 더 명확하게 드러냅니다.  
- 저차원으로 변환된 데이터는 시각화와 해석이 용이합니다.  
- 차원이 축소되면 데이터 처리와 분석이 더 빠르고 효율적으로 이루어질 수 있습니다.  
# 단점
- 주성분은 원본 변수들의 선형 결합으로 구성되므로 이들이 무엇을 의미하는지 해석하기 어려울 수 있습니다.
- 차원을 줄이는 과정에서 일부 정보가 손실될 수 있습니다.
- 모든 변수가 동일한 중요도를 가지고 있다고 가정하므로, 어떤 변수가 더 중요한지 구별하기 어렵습니다.


- PCA 시 결측치가 있을 경우 잘 못된 결과로 인해 해석이 안될 수 있다. 

```python
from sklearn.preprocessing import StandardScaler

columns_to_replace = ['Pregnancies', 'Glucose', 'BloodPressure', 
                      'SkinThickness', 'Insulin', 'BMI']

for col in columns_to_replace:
    mean_val = train[col].replace(0, np.nan).mean()
    train[col] = train[col].replace(0, mean_val)
    test[col] = test[col].replace(0, mean_val)


# 데이터에 대한 표준화 : 평균을 0, 표준편차를 1로 조정
scaler = StandardScaler()  
scaled_train = scaler.fit_transform(train)
scaled_test = scaler.transform(test)
```

# PCA 분해 방법

- 고유값 및 고유 벡터
- SVD(Singular Value Decomposition)


## SVD

SVD는 고유값 분해에 비해 수치적으로 더 안정적입니다.  
특히 고차원 데이터나 특이값이 매우 작은 경우에 이점이 있습니다.  
SVD는 고차원 데이터에 대해서도 효율적으로 작동하며,  
분산이 낮은 주성분을 빠르게 제거할 수 있습니다.  

```python
# SVD 수행
U, S, Vt = np.linalg.svd(scaled_train, full_matrices=False)
print(f'Vt의 shape는 {Vt.shape} 입니다.')
Vt
```

# PCA 설명 분산 비율(`explained_variance_ratio_`)의 의미
PCA 분석 후 각 주성분(Principal Component)이 원본 데이터의 전체 정보(분산)를 얼만큼 보존하여 설명하고 있는지를 나타내는 지표인 `explained_variance_ratio_`에 대해 설명합니다.

`pca.explained_variance_ratio_`는 PCA를 통해 만들어진 각각의 주성분(축)들이 기저 원본 데이터가 가진 **전체 정보량(분산, 데이터가 퍼져있는 정도)의 몇 퍼센트(%)를 담당하고 있는지**를 보여주는 비율 수치입니다.

### 쉽게 이해하는 해석 방법
만약 데이터가 100개의 컬럼(차원)으로 이루어져 있고, 이를 PCA를 통해 2개의 주성분(제1주성분, 제2주성분)으로 줄였다고 가정해 보겠습니다. 

```python
print(pca.explained_variance_ratio_)
# 출력 결과 예시: [0.60, 0.25]
```

- **첫 번째 숫자 (0.60):** 
  가장 먼저 만들어진 '제1주성분(PC1)'이 원본 데이터 전체 정보의 **60%**를 설명(대변)한다는 뜻입니다.
- **두 번째 숫자 (0.25):** 
  그다음으로 만들어진 '제2주성분(PC2)'이 남은 정보 중 **25%**를 설명한다는 뜻입니다.
- **누적 비율 (0.60 + 0.25 = 0.85):** 
  원본 데이터의 100개 컬럼을 고작 2개의 컬럼으로 압축했지만, 우리는 여전히 원본 데이터가 가진 특성의 **85%**를 보존하고 있다는 의미가 됩니다.

차원을 줄일 때는 보통 이 누적 설명 비율(Cumulative Explained Variance)이 **70~90%** 정도가 되도록 주성분 갯수(`n_components`)를 맞추는 것이 정석입니다. 

### 관련 함수 사용법
```python
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(X_train)

# 각 주성분이 설명하는 비율 출력
print("각 주성분의 설명 분산 비율:", pca.explained_variance_ratio_)

# 전체 보존된 정보량(누적합) 확인
print("총 보존된 정보량(비율):", sum(pca.explained_variance_ratio_))
```

### 추가 자료
- [Scikit-learn 공식 문서 (PCA - explained_variance_ratio_)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)